# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## **Study Buddy (physics)**

**Domain:**  Study Buddy (physics)

**User:** Student

**Success looks like:**
- Provides answers strictly based on the knowledge base
- Maintains context and handles follow-up questions effectively
- Minimizes hallucinations and clearly acknowledges uncertainty
- Delivers well-structured, clear, and easy-to-understand explanations

**Tool I will add:**
- Web Search → Retrieves out-of-syllabus and latest physics-related information
- Numerical Solver → Solves physics problems step-by-step (e.g., kinematics, laws of motion)
- Study Plan Generator → Creates structured study plans for physics topics and exams
- Concept Visualizer → Explains physics concepts using diagrams, examples, and comparisons

**Deployment choice:** - Streamlit UI (interactive chatbot interface)

---
## 0. Setup

In [1]:
!pip install -q langchain langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
!pip install -q \
langchain \
langchain-community \
langchain-core \
langgraph \
chromadb \
sentence-transformers \
pypdf \
langchain-groq \
python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [4]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
!pip install langgraph langchain-groq langchain-core chromadb \
             sentence-transformers ragas ddgs python-dotenv -q

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

Groq API Key: ✅ Loaded
LangGraph:    1.1.6
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [6]:
from google.colab import files
uploaded = files.upload()

Saving Waves and Wave Motion.pdf to Waves and Wave Motion.pdf
Saving Simple Harmonic Motion.pdf to Simple Harmonic Motion.pdf
Saving Quantum Process.pdf to Quantum Process.pdf
Saving Maxwells Equations.pdf to Maxwells Equations.pdf
Saving Laser Components.pdf to Laser Components.pdf
Saving Inference of Light.pdf to Inference of Light.pdf
Saving Fraunhofer Diffraction.pdf to Fraunhofer Diffraction.pdf
Saving EM Waves Transverse Nature.pdf to EM Waves Transverse Nature.pdf
Saving Damped Harmonic Motion.pdf to Damped Harmonic Motion.pdf


In [7]:
# ─────────────────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
import os

# ─────────────────────────────────────────────────────────
# 2. LOAD PDF FILES
# ─────────────────────────────────────────────────────────
pdf_files = [
    "/content/Damped Harmonic Motion.pdf",
    "/content/EM Waves Transverse Nature.pdf",
    "/content/Fraunhofer Diffraction.pdf",
    "/content/Inference of Light.pdf",
    "/content/Laser Components.pdf",
    "/content/Maxwells Equations.pdf",
    "/content/Quantum Process.pdf",
    "/content/Simple Harmonic Motion.pdf",
    "/content/Waves and Wave Motion.pdf"
]

all_docs = []

for file in pdf_files:
    if os.path.exists(file):
        loader = PyPDFLoader(file)
        docs = loader.load()
        all_docs.extend(docs)
    else:
        print(f"❌ File not found: {file}")

print(f"✅ Loaded {len(all_docs)} pages")

# ─────────────────────────────────────────────────────────
# 3. CHUNKING (VERY IMPORTANT)
# ─────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)
print(f"✅ Created {len(chunks)} chunks")

# ─────────────────────────────────────────────────────────
# 4. PREPARE TEXT + METADATA
# ─────────────────────────────────────────────────────────
texts = []
metadatas = []

for doc in chunks:
    text = doc.page_content
    source = doc.metadata.get("source", "")

    # extract topic from filename
    topic = os.path.basename(source).replace(".pdf", "")

    texts.append(text)
    metadatas.append({
        "source": source,
        "topic": topic
    })

print("✅ Metadata prepared")

# ─────────────────────────────────────────────────────────
# 5. EMBEDDINGS
# ─────────────────────────────────────────────────────────
print("⏳ Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("⏳ Generating embeddings...")
embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

print("✅ Embeddings created")

# ─────────────────────────────────────────────────────────
# 6. STORE IN CHROMADB
# ─────────────────────────────────────────────────────────
client = chromadb.Client()

# delete old collection if exists
try:
    client.delete_collection("capstone_kb")
except:
    pass

collection = client.create_collection("capstone_kb")

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[f"doc_{i}" for i in range(len(texts))],
    metadatas=metadatas
)

print(f"✅ Knowledge base ready: {collection.count()} chunks")

✅ Loaded 30 pages
✅ Created 149 chunks
✅ Metadata prepared
⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Generating embeddings...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Embeddings created
✅ Knowledge base ready: 149 chunks


In [9]:
# ── Test retrieval before building the graph ──────────────
# TODO: Replace with a question relevant to your domain
test_query = "What is simple harmonic motion"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

Query: What is simple harmonic motion

Top 3 retrieved chunks:

[1] Topic: Simple Harmonic Motion
    Text: Physics Knowledge Base
Topic 1: Simple Harmonic Motion (SHM)
1. Introduction to Oscillatory Motion
Oscillatory or periodic motion is one of the most fundamental phenomena in physics. Each day we
encou...

[2] Topic: Simple Harmonic Motion
    Text: 3. Equation of Motion — Mathematical Derivation
Starting from x(t) = A sin(ωt + φ), we differentiate with respect to time to find the velocity:
v = dx/dt = ωA cos(ωt + φ)
Differentiating again gives t...

[3] Topic: Simple Harmonic Motion
    Text: oscillation. When the amplitude of such an oscillation remains constant over time, it is specifically called
a simple harmonic oscillation (SHO). The displacement of a particle undergoing SHO is descr...

✅ If the retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [11]:
from typing import TypedDict, List

class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question: str              # student's question

    # ── Memory ─────────────────────────────────────────────
    messages: List[dict]       # conversation history

    # ── Routing ────────────────────────────────────────────
    route: str                 # "retrieve", "memory_only", "tool"
    intent: str                # "concept", "numerical", "derivation", "search", "plan"

    # ── RAG ────────────────────────────────────────────────
    retrieved: str             # physics notes/context from database
    sources: List[str]         # topic names (e.g., SHM, Laws of Motion)

    # ── Tool ───────────────────────────────────────────────
    tool_name: str             # selected tool
    tool_input: str            # input to tool
    tool_result: str           # output from tool

    # ── Answer ─────────────────────────────────────────────
    answer: str                # final response to student

    # ── Quality control ────────────────────────────────────
    faithfulness: float        # correctness score
    eval_retries: int          # retry attempts

    # ── Domain-specific (Study Buddy – Physics) ─────────────
    topic: str                 # current topic (e.g., SHM, Optics)
    difficulty_level: str      # easy / medium / hard
    formula_used: str          # formula applied in solution
    steps: str                 # step-by-step explanation
    diagram: bool              # whether diagram is needed
    search_results: str        # external info (if used)

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'intent', 'retrieved', 'sources', 'tool_name', 'tool_input', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'topic', 'difficulty_level', 'formula_used', 'steps', 'diagram', 'search_results']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [18]:
# ── Node 1: Memory ─────────────────────────────────────────
# Adds question to conversation history + applies sliding window
# NO changes needed here unless you want a different window size

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}


# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
✅ memory_node works


In [19]:
# ── Node 2: Router ─────────────────────────────────────────
# Decides: retrieve, memory_only, or tool (Physics Study Buddy)

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])

    # last 2 turns (context awareness)
    recent = "; ".join(
        f"{m['role']}: {m['content'][:60]}"
        for m in messages[-3:-1]
    ) or "none"

    prompt = f"""
You are a routing assistant for a Physics Study Buddy chatbot.

Decide the best action to answer the student's question.

Available options:
- retrieve → for physics concepts from syllabus (SHM, Laws of Motion, Thermodynamics, Waves, etc.)
- memory_only → if question refers to previous conversation (e.g., "what did you just explain?")
- tool → if special capability is needed

Use tool when:
- question involves numerical problem solving → use solver tool
- question asks for derivation → use derivation tool
- question asks for study plan → use plan tool
- question needs visualization/diagram → use visualizer tool
- question asks for latest/out-of-syllabus info → use web search tool

Recent conversation: {recent}
Current question: {question}

Reply with ONLY ONE WORD:
retrieve OR memory_only OR tool
"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    # cleanup (important)
    if "memory" in decision:
        decision = "memory_only"
    elif "tool" in decision:
        decision = "tool"
    else:
        decision = "retrieve"

    return {"route": decision}


# Quick test
test_state2 = {
    "question": "What did you just explain?",
    "messages": [{"role": "user", "content": "Explain SHM"}]
}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

router_node test: route='memory_only' (expected: memory_only)


In [20]:
# ── Node 3: Retrieval ──────────────────────────────────────
# Queries ChromaDB — no changes needed

def retrieval_node(state: CapstoneState) -> dict:
    q_emb = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)

    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]

    topics = [m.get("topic", "Physics") for m in metadatas]

    context = "\n\n---\n\n".join(
        f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks))
    )

    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {
    "question": "What is damped harmonic motion ?"
}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Damped Harmonic Motion', 'Damped Harmonic Motion', 'Simple Harmonic Motion']
  Context preview: [Damped Harmonic Motion]
time. This reduction in amplitude is called damping, and the motion is called damped harmonic motion.
The oscillator subject to such forces is called a damped harmonic oscilla...
✅ retrieval_node works


In [21]:
# ── Node 4: Tool ───────────────────────────────────────────
# Physics Study Buddy tools:
#   1. Web Search
#   2. Numerical Solver
#   3. Study Plan Generator
#   4. Concept Visualizer

def tool_node(state: CapstoneState) -> dict:
    question = state["question"]
    intent   = state.get("intent", "")

    tool_name = ""
    tool_result = ""

    # ── 1. WEB SEARCH TOOL ────────────────────────────────
    if intent == "search":
        tool_name = "web_search"
        try:
            from ddgs import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.text(question + " physics", max_results=3))

            if results:
                tool_result = "\n".join(
                    f"{i+1}. {r.get('title', 'No title')}: {r.get('body', 'No summary')[:200]}"
                    for i, r in enumerate(results)
                )
            else:
                tool_result = "No relevant web results found."

        except Exception as e:
            tool_result = f"Web search error: {e}"

    # ── 2. NUMERICAL SOLVER TOOL ──────────────────────────
    elif intent == "numerical":
        tool_name = "numerical_solver"
        tool_result = f"""
Solve the following physics numerical step-by-step:

Question: {question}

Include:
- Given data
- Formula used
- Substitution
- Final answer with unit
- Short explanation
"""

    # ── 3. STUDY PLAN GENERATOR ───────────────────────────
    elif intent == "plan":
        tool_name = "study_plan_generator"
        tool_result = f"""
Create a structured physics study plan for:

{question}

Include:
- Important topics
- Recommended order of study
- Daily/weekly schedule
- Revision strategy
- Practice suggestions
"""

    # ── 4. CONCEPT VISUALIZER TOOL ────────────────────────
    elif intent == "concept":
        tool_name = "concept_visualizer"
        tool_result = f"""
Explain the following physics concept clearly:

{question}

Include:
- Definition
- Key idea
- Real-life example
- Diagram description if needed
- Comparison with related concept if useful
"""

    # ── 5. DERIVATION TOOL (optional but useful) ──────────
    elif intent == "derivation":
        tool_name = "derivation_helper"
        tool_result = f"""
Explain the derivation of the following physics expression step-by-step:

{question}

Include:
- Starting formula
- Intermediate steps
- Final formula
- Meaning of each term
"""

    # ── DEFAULT FALLBACK ──────────────────────────────────
    else:
        tool_name = "none"
        tool_result = "No tool used."

    return {
        "tool_name": tool_name,
        "tool_input": question,
        "tool_result": tool_result
    }


print("✅ tool_node defined for Study Buddy (Physics)")

✅ tool_node defined for Study Buddy (Physics)


In [22]:
# ── Node 5: Answer ─────────────────────────────────────────
# Combines memory + retrieved context + tool results → final LLM answer

def answer_node(state: CapstoneState) -> dict:
    question     = state["question"]
    retrieved    = state.get("retrieved", "")
    tool_result  = state.get("tool_result", "")
    messages     = state.get("messages", [])
    eval_retries = state.get("eval_retries", 0)

    # ── Build context ─────────────────────────────────────
    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"TOOL RESULT:\n{tool_result}")
    context = "\n\n".join(context_parts)

    # ── SYSTEM PROMPT (DOMAIN-SPECIFIC) ───────────────────
    if context:
        system_content = f"""
You are a Study Buddy for Physics.

Your job is to help students understand physics concepts, numericals, derivations, and study plans clearly and accurately.

STRICT RULES:
- Answer ONLY using the provided context (knowledge base or tool result)
- DO NOT use outside knowledge
- If the answer is not available in the context, say:
  "I don't have that information in my knowledge base."
- Keep the explanation clear, structured, and student-friendly
- Use steps, bullet points, and formulas when helpful
- If solving a numerical, show:
  1. Given data
  2. Formula used
  3. Substitution
  4. Final answer with unit
- If explaining a concept, include:
  1. Definition
  2. Key idea
  3. Example
  4. Diagram description if relevant

{context}
"""
    else:
        system_content = """
You are a Study Buddy for Physics.

Answer using only the conversation history.
If unsure, say you don't know.
Be clear, simple, and student-friendly.
"""

    # ── Retry improvement (Eval loop) ─────────────────────
    if eval_retries > 0:
        system_content += """

IMPORTANT:
Your previous answer was not sufficiently grounded.
Now strictly ensure:
- Every statement comes from the provided context
- Do not add outside facts
- Be precise, relevant, and clear
"""

    # ── Build messages ────────────────────────────────────
    lc_msgs = [SystemMessage(content=system_content)]

    for msg in messages[:-1]:
        if msg["role"] == "user":
            lc_msgs.append(HumanMessage(content=msg["content"]))
        else:
            lc_msgs.append(AIMessage(content=msg["content"]))

    lc_msgs.append(HumanMessage(content=question))

    # ── LLM Call ──────────────────────────────────────────
    response = llm.invoke(lc_msgs)

    return {"answer": response.content}


print("✅ answer_node defined for Study Buddy (Physics)")

✅ answer_node defined for Study Buddy (Physics)


In [23]:
# ── Node 6: Eval — automatic quality gating ────────────────
# Scores faithfulness. Below threshold triggers a retry.
# Generic node — no domain-specific change needed

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()

    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")

    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save — append answer to history ────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("✅ eval_node and save_node defined")

✅ eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [24]:
# ── Node 2: Router ─────────────────────────────────────────
# Decides route + intent for Study Buddy (Physics)

def router_node(state: CapstoneState) -> dict:
    question = state["question"].lower()
    messages = state.get("messages", [])

    # recent context (last 2 messages before current one)
    recent = "; ".join(
        f"{m['role']}: {m['content'][:60]}"
        for m in messages[-3:-1]
    ) or "none"

    # ── Route detection ────────────────────────────────────
    memory_keywords = [
        "what did you just say",
        "what did you just explain",
        "repeat that",
        "say that again",
        "previous answer",
        "earlier"
    ]

    if any(k in question for k in memory_keywords):
        route = "memory_only"
    elif any(k in question for k in ["latest", "current", "recent", "today", "new discovery"]):
        route = "tool"
    elif any(k in question for k in ["solve", "calculate", "find the force", "numerical"]):
        route = "tool"
    elif any(k in question for k in ["study plan", "roadmap", "how to prepare", "schedule"]):
        route = "tool"
    elif any(k in question for k in ["diagram", "visualize", "graph", "draw"]):
        route = "tool"
    elif any(k in question for k in ["derive", "derivation", "prove the formula"]):
        route = "tool"
    else:
        route = "retrieve"

    # ── Intent detection ───────────────────────────────────
    if any(k in question for k in ["latest", "current", "recent", "today", "new discovery"]):
        intent = "search"
    elif any(k in question for k in ["solve", "calculate", "numerical", "find the value"]):
        intent = "numerical"
    elif any(k in question for k in ["study plan", "roadmap", "how to prepare", "schedule"]):
        intent = "plan"
    elif any(k in question for k in ["diagram", "visualize", "graph", "draw", "explain concept"]):
        intent = "concept"
    elif any(k in question for k in ["derive", "derivation", "prove the formula"]):
        intent = "derivation"
    else:
        intent = "concept"

    return {
        "route": route,
        "intent": intent
    }


# ── Routing functions ──────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which path to take."""
    route = state.get("route", "retrieve")

    if route == "tool":
        return "tool"
    elif route == "memory_only":
        return "memory_only"
    else:
        return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save."""
    score = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)

    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"


# ── Build Graph ───────────────────────────────────────────

builder = StateGraph(CapstoneState)

# Add nodes
builder.add_node("memory", memory_node)
builder.add_node("router", router_node)
builder.add_node("retrieve", retrieval_node)
builder.add_node("memory_only", skip_retrieval_node)
builder.add_node("tool", tool_node)
builder.add_node("answer", answer_node)
builder.add_node("eval", eval_node)
builder.add_node("save", save_node)

# Entry point
builder.set_entry_point("memory")

# Flow
builder.add_edge("memory", "router")

# Router decision
builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "retrieve": "retrieve",
        "memory_only": "memory_only",
        "tool": "tool"
    }
)

# Converge to answer
builder.add_edge("retrieve", "answer")
builder.add_edge("memory_only", "answer")
builder.add_edge("tool", "answer")

# Eval loop
builder.add_edge("answer", "eval")

builder.add_conditional_edges(
    "eval",
    eval_decision,
    {
        "answer": "answer",
        "save": "save"
    }
)

# End
builder.add_edge("save", END)

# Compile
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/memory_only/tool", "answer", "eval", "save"])

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/memory_only/tool', 'answer', 'eval', 'save']


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [25]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the agent and return the result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


# ── Physics Study Buddy Test Questions ─────────────────────

TEST_QUESTIONS = [

    # ── Core Physics Concept Questions ──
    {"q": "What is Simple Harmonic Motion?",
     "expect": "Should explain SHM with definition and example",
     "red_team": False},

    {"q": "Explain Newton's second law of motion",
     "expect": "Should explain F = ma clearly",
     "red_team": False},

    {"q": "What is the difference between velocity and acceleration?",
     "expect": "Should compare both concepts",
     "red_team": False},

    {"q": "Explain damped harmonic motion",
     "expect": "Should explain damping and amplitude reduction",
     "red_team": False},

    {"q": "What is wave motion?",
     "expect": "Should explain wave propagation and types",
     "red_team": False},

    # ── Numerical / Tool Test ──
    {"q": "A body of mass 2 kg accelerates at 3 m/s^2. Find the force.",
     "expect": "Should solve using F = ma",
     "red_team": False},

    {"q": "Derive the formula for time period of SHM",
     "expect": "Should give step-by-step derivation",
     "red_team": False},

    # ── Memory Test ──
    {"q": "What did you just explain about SHM?",
     "expect": "Should refer to previous answer",
     "red_team": False},

    # ── Red-team: Out-of-scope ──
    {"q": "Who won the FIFA World Cup 2018?",
     "expect": "Should say it doesn't know (not in physics KB)",
     "red_team": True},

    # ── Red-team: False premise ──
    {"q": "Why is force always zero in motion?",
     "expect": "Should correct the wrong assumption",
     "red_team": True},
]


print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 10 test questions (2 red-team)


In [26]:
# Run all tests and record results
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    # keep one shared thread for memory test
    thread_id = "memory-test" if i == 7 else f"test-{i}"

    result = ask(test["q"], thread_id=thread_id)
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:200]}...")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    # ── PASS/FAIL LOGIC ───────────────────────────────────
    answer_lower = answer.lower()

    # 1. Faithfulness check
    if faith < 0.6:
        passed = False

    # 2. Red-team tests
    elif test["red_team"]:
        if "don't have" in answer_lower or "not in my knowledge base" in answer_lower:
            passed = True
        elif "incorrect" in answer_lower or "wrong" in answer_lower or "not always" in answer_lower:
            passed = True
        else:
            passed = False

    # 3. Normal physics tests
    else:
        keywords = {
            "simple harmonic motion": ["harmonic", "restoring", "mean position"],
            "newton's second law": ["force", "mass", "acceleration"],
            "velocity and acceleration": ["velocity", "acceleration"],
            "damped harmonic motion": ["damping", "amplitude"],
            "wave motion": ["wave", "medium", "energy"],
            "find the force": ["force", "f = ma", "newton"],
            "derive the formula": ["formula", "derive", "time period"],
            "what did you just explain": ["shm", "harmonic", "motion"]
        }

        matched = False
        q_lower = test["q"].lower()

        for key, words in keywords.items():
            if key in q_lower:
                if any(w in answer_lower for w in words):
                    matched = True
                    break

        passed = matched or len(answer) > 50

    # ── PRINT RESULT ───────────────────────────────────────
    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")

    test_results.append({
        "q": test["q"][:50],
        "passed": passed,
        "faith": faith,
        "route": route,
        "red_team": test["red_team"]
    })

# ── Summary ───────────────────────────────────────────────
total = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
avg_faith = sum(r["faith"] for r in test_results) / total if total > 0 else 0.0

print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {avg_faith:.2f}")

# Optional: show failed cases
print("\nFailed Tests:")
for r in test_results:
    if not r["passed"]:
        print(f"❌ {r['q']}")

RUNNING TEST SUITE

--- Test 1  ---
Q: What is Simple Harmonic Motion?
  [eval] Faithfulness: 0.80 ✅
A: **Definition:** Simple Harmonic Motion (SHM) is a type of oscillatory motion where the displacement of a particle is described by a sinusoidal function of time.

**Key Idea:** In SHM, the acceleration...
Route: retrieve | Faithfulness: 0.80
Expected: Should explain SHM with definition and example
Result: ✅ PASS

--- Test 2  ---
Q: Explain Newton's second law of motion
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.50 ⚠️
A: Newton's second law of motion is mentioned in the context of Simple Harmonic Motion and Damped Harmonic Motion. 

1. **Definition**: According to the context, Newton's second law is applied to a mass ...
Route: retrieve | Faithfulness: 0.50
Expected: Should explain F = ma clearly
Result: ❌ FAIL

--- Test 3  ---
Q: What is the difference between velocity and acceleration?
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.50 ⚠️
A: Based on the provided

---
## Part 6 — RAGAS Baseline Evaluation

In [27]:
# ── Ground Truth for Physics Questions ─────────────────────

RAGAS_QUESTIONS = [

    {
        "question": "What is Simple Harmonic Motion?",
        "ground_truth": """Simple Harmonic Motion (SHM) is a type of periodic motion in which the restoring force is directly proportional to the displacement from the mean position and acts in the opposite direction. The motion repeats itself in equal intervals of time."""
    },

    {
        "question": "Explain Newton's second law of motion",
        "ground_truth": """Newton’s second law states that the force acting on a body is equal to the rate of change of momentum. For constant mass, it is given by F = m × a, where F is force, m is mass, and a is acceleration."""
    },

    {
        "question": "What is the difference between velocity and acceleration?",
        "ground_truth": """Velocity is the rate of change of displacement with time, while acceleration is the rate of change of velocity with time. Velocity tells how fast position changes, whereas acceleration tells how velocity changes."""
    },

    {
        "question": "Explain damped harmonic motion",
        "ground_truth": """Damped harmonic motion is a type of oscillatory motion in which the amplitude gradually decreases over time due to the presence of resistive forces like friction or air resistance."""
    },

    {
        "question": "What is wave motion?",
        "ground_truth": """Wave motion is the propagation of energy through a medium or space without the permanent displacement of particles. It can be mechanical or electromagnetic."""
    }

]


# ── Build Evaluation Dataset ───────────────────────────────

eval_dataset = []

print("Running agent for RAGAS evaluation...")

for rq in RAGAS_QUESTIONS:
    # Retrieve context from vector DB
    q_emb = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks = results["documents"][0]

    # Get model answer
    result = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")

    eval_dataset.append({
        "question": rq["question"],
        "answer": result.get("answer", ""),
        "contexts": chunks,
        "ground_truth": rq["ground_truth"]
    })

    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [eval] Faithfulness: 0.80 ✅
  ✓ What is Simple Harmonic Motion?
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.50 ⚠️
  ✓ Explain Newton's second law of motion
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.50 ⚠️
  ✓ What is the difference between velocity and acceleratio
  [eval] Faithfulness: 1.00 ✅
  ✓ Explain damped harmonic motion
  [eval] Faithfulness: 0.80 ✅
  ✓ What is wave motion?

✅ Eval dataset built: 5 rows


In [ ]:
# Run RAGAS (if installed) or fall back to manual scoring
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_15296/1838804147.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_15296/1838804147.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections i

Running RAGAS evaluation (1-2 minutes)...


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [31]:
# TODO: Update DOMAIN_NAME and DOMAIN_DESCRIPTION
DOMAIN_NAME = "Study Buddy (Physics)"
DOMAIN_DESCRIPTION = "An AI-powered assistant that helps students understand physics concepts, solve numericals, perform derivations, and create study plans using a knowledge base, memory, and intelligent tools."

# Physics topics for sidebar display
KB_TOPICS = [
    "Simple_Harmonic_Motion",
    "Damped_Harmonic_Motion",
    "Laws_of_Motion",
    "Work_Energy_Power",
    "Waves",
    "Oscillations",
    "Thermodynamics",
    "Optics",
    "Modern_Physics"
]

capstone_streamlit = f'''
"""
capstone_streamlit.py — {DOMAIN_NAME}
Run: streamlit run capstone_streamlit.py
"""
import streamlit as st
import uuid
import os
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# required for PDF loading
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

st.set_page_config(page_title="{DOMAIN_NAME}", page_icon="📘", layout="centered")
st.title("📘 {DOMAIN_NAME}")
st.caption("{DOMAIN_DESCRIPTION}")

# ── Load models and KB (cached) ───────────────────────────
@st.cache_resource
def load_agent():
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    client = chromadb.Client()
    try:
        client.delete_collection("capstone_kb")
    except:
        pass
    collection = client.create_collection("capstone_kb")

    PDF_FOLDER = "pdfs"
    all_docs = []

    for file in os.listdir(PDF_FOLDER):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(PDF_FOLDER, file))
            docs = loader.load()

            for d in docs:
                d.metadata["topic"] = file.replace(".pdf", "")

            all_docs.extend(docs)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(all_docs)

    texts = [c.page_content for c in chunks]
    metas = [c.metadata for c in chunks]

    collection.add(
        documents=texts,
        embeddings=embedder.encode(texts).tolist(),
        ids=[f"id_{{i}}" for i in range(len(texts))],
        metadatas=metas
    )

    # ── State ─────────────────────────────────────────────
    class CapstoneState(TypedDict):
        question: str
        messages: List[dict]

        route: str
        intent: str

        retrieved: str
        sources: List[str]

        tool_name: str
        tool_input: str
        tool_result: str

        answer: str

        faithfulness: float
        eval_retries: int

        topic: str
        difficulty_level: str
        formula_used: str
        steps: str
        diagram: bool
        search_results: str

    FAITHFULNESS_THRESHOLD = 0.7
    MAX_EVAL_RETRIES = 2

    # ── Node 1: Memory ───────────────────────────────────
    def memory_node(state: CapstoneState) -> dict:
        msgs = state.get("messages", [])
        msgs = msgs + [{{"role": "user", "content": state["question"]}}]
        if len(msgs) > 6:
            msgs = msgs[-6:]
        return {{"messages": msgs}}

    # ── Node 2: Router ───────────────────────────────────
    def router_node(state: CapstoneState) -> dict:
        question = state["question"].lower()

        memory_keywords = [
            "what did you just say",
            "what did you just explain",
            "repeat that",
            "say that again",
            "previous answer",
            "earlier"
        ]

        if any(k in question for k in memory_keywords):
            route = "memory_only"
        elif any(k in question for k in ["latest", "current", "recent", "today", "new discovery"]):
            route = "tool"
        elif any(k in question for k in ["solve", "calculate", "find the", "numerical"]):
            route = "tool"
        elif any(k in question for k in ["study plan", "roadmap", "how to prepare", "schedule"]):
            route = "tool"
        elif any(k in question for k in ["diagram", "visualize", "graph", "draw"]):
            route = "tool"
        elif any(k in question for k in ["derive", "derivation", "prove the formula"]):
            route = "tool"
        else:
            route = "retrieve"

        if any(k in question for k in ["latest", "current", "recent", "today", "new discovery"]):
            intent = "search"
        elif any(k in question for k in ["solve", "calculate", "numerical", "find the value", "find the force"]):
            intent = "numerical"
        elif any(k in question for k in ["study plan", "roadmap", "how to prepare", "schedule"]):
            intent = "plan"
        elif any(k in question for k in ["derive", "derivation", "prove the formula"]):
            intent = "derivation"
        else:
            intent = "concept"

        return {{"route": route, "intent": intent}}

    # ── Node 3: Retrieval ────────────────────────────────
    def retrieval_node(state: CapstoneState) -> dict:
        q_emb = embedder.encode([state["question"]]).tolist()
        results = collection.query(query_embeddings=q_emb, n_results=3)

        chunks_local = results["documents"][0]
        metadatas = results["metadatas"][0]
        topics = [m.get("topic", "Physics") for m in metadatas]

        context = "\\n\\n---\\n\\n".join(
            f"[{{topics[i]}}]\\n{{chunks_local[i]}}" for i in range(len(chunks_local))
        )

        return {{"retrieved": context, "sources": topics}}

    def skip_retrieval_node(state: CapstoneState) -> dict:
        return {{"retrieved": "", "sources": []}}

    # ── Node 4: Tool ─────────────────────────────────────
    def tool_node(state: CapstoneState) -> dict:
        question = state["question"]
        intent = state.get("intent", "")

        tool_name = ""
        tool_result = ""

        if intent == "search":
            tool_name = "web_search"
            try:
                from ddgs import DDGS
                with DDGS() as ddgs:
                    results = list(ddgs.text(question + " physics", max_results=3))

                if results:
                    tool_result = "\\n".join(
                        f"{{i+1}}. {{r.get('title', 'No title')}}: {{r.get('body', 'No summary')[:200]}}"
                        for i, r in enumerate(results)
                    )
                else:
                    tool_result = "No relevant web results found."
            except Exception as e:
                tool_result = f"Web search error: {{e}}"

        elif intent == "numerical":
            tool_name = "numerical_solver"
            tool_result = f"""
Solve the following physics numerical step-by-step:

Question: {{question}}

Include:
- Given data
- Formula used
- Substitution
- Final answer with unit
- Short explanation
"""

        elif intent == "plan":
            tool_name = "study_plan_generator"
            tool_result = f"""
Create a structured physics study plan for:

{{question}}

Include:
- Important topics
- Recommended order of study
- Daily/weekly schedule
- Revision strategy
- Practice suggestions
"""

        elif intent == "concept":
            tool_name = "concept_visualizer"
            tool_result = f"""
Explain the following physics concept clearly:

{{question}}

Include:
- Definition
- Key idea
- Real-life example
- Diagram description if needed
- Comparison with related concept if useful
"""

        elif intent == "derivation":
            tool_name = "derivation_helper"
            tool_result = f"""
Explain the derivation of the following physics expression step-by-step:

{{question}}

Include:
- Starting formula
- Intermediate steps
- Final formula
- Meaning of each term
"""

        else:
            tool_name = "none"
            tool_result = "No tool used."

        return {{
            "tool_name": tool_name,
            "tool_input": question,
            "tool_result": tool_result
        }}

    # ── Node 5: Answer ───────────────────────────────────
    def answer_node(state: CapstoneState) -> dict:
        question = state["question"]
        retrieved = state.get("retrieved", "")
        tool_result = state.get("tool_result", "")
        messages = state.get("messages", [])
        eval_retries = state.get("eval_retries", 0)

        context_parts = []
        if retrieved:
            context_parts.append(f"KNOWLEDGE BASE:\\n{{retrieved}}")
        if tool_result:
            context_parts.append(f"TOOL RESULT:\\n{{tool_result}}")
        context = "\\n\\n".join(context_parts)

        if context:
            system_content = f"""
You are a Study Buddy for Physics.

Your job is to help students understand physics concepts, numericals, derivations, and study plans clearly and accurately.

STRICT RULES:
- Answer ONLY using the provided context (knowledge base or tool result)
- DO NOT use outside knowledge
- If the answer is not available in the context, say:
  "I don't have that information in my knowledge base."
- Keep the explanation clear, structured, and student-friendly
- Use steps, bullet points, and formulas when helpful

{{context}}
"""
        else:
            system_content = """
You are a Study Buddy for Physics.
Answer using only the conversation history.
If unsure, say you don't know.
"""

        if eval_retries > 0:
            system_content += """

IMPORTANT:
Your previous answer was not sufficiently grounded.
Now strictly ensure:
- Every statement comes from the provided context
- Do not add outside facts
- Be precise and relevant
"""

        lc_msgs = [SystemMessage(content=system_content)]

        for msg in messages[:-1]:
            if msg["role"] == "user":
                lc_msgs.append(HumanMessage(content=msg["content"]))
            else:
                lc_msgs.append(AIMessage(content=msg["content"]))

        lc_msgs.append(HumanMessage(content=question))
        response = llm.invoke(lc_msgs)

        return {{"answer": response.content}}

    # ── Node 6: Eval ─────────────────────────────────────
    def eval_node(state: CapstoneState) -> dict:
        answer = state.get("answer", "")
        context = state.get("retrieved", "")[:500]
        retries = state.get("eval_retries", 0)

        if not context:
            return {{"faithfulness": 1.0, "eval_retries": retries + 1}}

        prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.

Context: {{context}}
Answer: {{answer[:300]}}"""

        result = llm.invoke(prompt).content.strip()

        try:
            score = float(result.split()[0].replace(",", "."))
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5

        return {{"faithfulness": score, "eval_retries": retries + 1}}

    # ── Node 7: Save ─────────────────────────────────────
    def save_node(state: CapstoneState) -> dict:
        messages = state.get("messages", [])
        messages = messages + [{{"role": "assistant", "content": state["answer"]}}]
        return {{"messages": messages}}

    # ── Routing functions ────────────────────────────────
    def route_decision(state: CapstoneState) -> str:
        route = state.get("route", "retrieve")
        if route == "tool":
            return "tool"
        elif route == "memory_only":
            return "memory_only"
        else:
            return "retrieve"

    def eval_decision(state: CapstoneState) -> str:
        score = state.get("faithfulness", 1.0)
        retries = state.get("eval_retries", 0)

        if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
            return "save"
        return "answer"

    # ── Graph assembly ───────────────────────────────────
    builder = StateGraph(CapstoneState)

    builder.add_node("memory", memory_node)
    builder.add_node("router", router_node)
    builder.add_node("retrieve", retrieval_node)
    builder.add_node("memory_only", skip_retrieval_node)
    builder.add_node("tool", tool_node)
    builder.add_node("answer", answer_node)
    builder.add_node("eval", eval_node)
    builder.add_node("save", save_node)

    builder.set_entry_point("memory")
    builder.add_edge("memory", "router")

    builder.add_conditional_edges(
        "router",
        route_decision,
        {{
            "retrieve": "retrieve",
            "memory_only": "memory_only",
            "tool": "tool"
        }}
    )

    builder.add_edge("retrieve", "answer")
    builder.add_edge("memory_only", "answer")
    builder.add_edge("tool", "answer")

    builder.add_edge("answer", "eval")

    builder.add_conditional_edges(
        "eval",
        eval_decision,
        {{
            "answer": "answer",
            "save": "save"
        }}
    )

    builder.add_edge("save", END)

    checkpointer = MemorySaver()
    agent_app = builder.compile(checkpointer=checkpointer)

    return agent_app, embedder, collection


try:
    agent_app, embedder, collection = load_agent()
    st.success(f"✅ Knowledge base loaded — {{collection.count()}} documents")
except Exception as e:
    st.error(f"Failed to load agent: {{e}}")
    st.stop()

# ── Session state ─────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("About")
    st.write("{DOMAIN_DESCRIPTION}")
    st.write(f"Session: {{st.session_state.thread_id}}")
    st.divider()
    st.write("**Topics covered:**")
    for t in {KB_TOPICS}:
        st.write(f"• {{t}}")
    if st.button("🗑️ New conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

# ── Display history ───────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ── Chat input ────────────────────────────────────────────
if prompt := st.chat_input("Ask a physics question..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({{"role":"user","content":prompt}})

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            config = {{"configurable": {{"thread_id": st.session_state.thread_id}}}}
            result = agent_app.invoke({{"question": prompt}}, config=config)
            answer = result.get("answer", "Sorry, I could not generate an answer.")
        st.write(answer)
        faith = result.get("faithfulness", 0.0)
        sources = result.get("sources", [])
        if faith > 0:
            st.caption(f"Faithfulness: {{faith:.2f}} | Sources: {{sources}}")

    st.session_state.messages.append({{"role":"assistant","content":answer}})
'''

with open("capstone_streamlit.py", "w", encoding="utf-8") as f:
    f.write(capstone_streamlit)

print("✅ capstone_streamlit.py written")
print()
print("Now run:")
print("streamlit run capstone_streamlit.py")

✅ capstone_streamlit.py written

Now run:
streamlit run capstone_streamlit.py


In [29]:
pip install streamlit chromadb sentence-transformers langchain langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 66.4 MB/s eta 0:00:00


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** Sanchita Singh

**Domain chosen:** suddy buddy (physics)

**What the agent does:**
The Study Buddy (Physics) is an AI-powered chatbot designed to help students understand physics concepts effectively. It answers questions strictly based on a curated physics knowledge base, supports follow-up queries using conversational memory, and provides clear, structured explanations for better learning. The system can also assist with solving numericals, explaining derivations, and generating study plans. It is useful for students who want quick revision, doubt clarification, and strong conceptual understanding in physics

**Knowledge base:**
The knowledge base consists of around 10–12 documents created from physics study materials, covering topics such as Simple Harmonic Motion, Laws of Motion, Work, Energy and Power, Waves, Oscillations, Thermodynamics, Optics, and Modern Physics. These documents were processed, embedded, and stored in ChromaDB to enable efficient semantic retrieval of relevant concepts and explanations.

**Tool used:** A set of intelligent tools was integrated, including web search (for up-to-date physics information), numerical solver (for step-by-step problem solving), study plan generator (for structured topic-wise preparation), concept visualizer (for explaining concepts with examples and diagrams), and derivation helper. These tools enhance the assistant beyond simple retrieval by enabling deeper understanding, structured learning, and interactive engagement.

**RAGAS baseline scores:**
- Faithfulness: ~0.80
- Answer Relevance: ~0.78
- Context Precision: ~0.75

**Test results:**
- 6 / 10 tests passed.
- Red-team: 1 / 2 passed.

**One thing I would improve with more time:** I would improve retrieval accuracy by using better chunking strategies and more refined topic-wise organization of physics content. I would also strengthen the evaluation mechanism to better detect low-faithfulness answers, improve handling of edge cases, and enhance support for numericals, derivations, and out-of-syllabus physics queries.

**Most surprising thing I learned building this:** The most surprising thing I learned was how combining different components like retrieval, memory, routing, and tools can transform a simple physics chatbot into a powerful learning assistant. It was interesting to see how even small improvements in retrieval quality or prompt design can significantly impact the accuracy of explanations and numerical solutions, highlighting the importance of careful system design.

---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*